# Electricity Demand Forecasting
 AI Club Recruitment Submission

## 1. Imports

In [294]:
import pandas as pd
import numpy as np


In [295]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor


## 2. Load Data

In [296]:
demand = pd.read_excel("PGCB_date_power_demand.xlsx")

demand['datetime'] = pd.to_datetime(demand['datetime'])
demand = demand.sort_values('datetime')
demand = demand.set_index('datetime')

# numeric values
demand = demand.select_dtypes(include=['number'])

# resampling
demand = demand.resample('h').mean()
demand = demand.fillna(method='ffill')
demand = demand.fillna(0)


/tmp/ipykernel_1402/3697318725.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  demand = demand.fillna(method='ffill')


outlier cleaning

In [297]:
rolling_median = demand['demand_mw'].shift(1).rolling(24, min_periods=1).median()

residual = demand['demand_mw'] - rolling_median
threshold = 3 * residual.std()

outliers = np.abs(residual) > threshold

demand.loc[outliers, 'demand_mw'] = rolling_median[outliers]

In [298]:
data = demand.copy()

In [299]:

print(demand.shape)
print(demand.isna().sum().sum())

(89101, 13)
0


In [300]:
print(data.dtypes)

generation_mw           float64
demand_mw               float64
load_shedding           float64
gas                     float64
liquid_fuel             float64
coal                    float64
hydro                   float64
solar                   float64
wind                    float64
india_bheramara_hvdc    float64
india_tripura           float64
india_adani             float64
nepal                   float64
dtype: object


In [301]:
print(data[['demand_mw']].head())

                     demand_mw
datetime                      
2015-04-19 00:00:00     4821.0
2015-04-19 01:00:00     3612.0
2015-04-19 02:00:00     3727.0
2015-04-19 03:00:00     3632.0
2015-04-19 04:00:00     3641.0


In [302]:
econ = pd.read_csv("economic_full_1.csv")

# filter your country (X in your sample)
econ = econ[econ['Country Name'] == 'X']

# drop non-year columns
econ = econ.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])

# transpose
econ = econ.T
econ.index.name = 'year'
econ = econ.reset_index()

# convert year
econ['year'] = econ['year'].astype(int)

# convert numeric
econ = econ.apply(pd.to_numeric, errors='coerce')

# create single economic feature
econ['econ_index'] = econ.drop(columns=['year']).mean(axis=1)

econ = econ[['year', 'econ_index']]

In [303]:
data['year'] = data.index.year

data = data.reset_index()
data = data.merge(econ, on='year', how='left')
data = data.set_index('datetime')

data = data.fillna(method='ffill')

/tmp/ipykernel_1402/3897597388.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [304]:
print(data[['econ_index']].head())

                       econ_index
datetime                         
2015-04-19 00:00:00  3.520954e+11
2015-04-19 01:00:00  3.520954e+11
2015-04-19 02:00:00  3.520954e+11
2015-04-19 03:00:00  3.520954e+11
2015-04-19 04:00:00  3.520954e+11


FEATURE ENGINEERING

In [305]:
data['hour'] = data.index.hour
data['dayofweek'] = data.index.dayofweek
data['month'] = data.index.month
data['is_weekend'] = (data['dayofweek'] >= 5).astype(int)

In [306]:
data['lag_1'] = data['demand_mw'].shift(1)
data['lag_2'] = data['demand_mw'].shift(2)
data['lag_24'] = data['demand_mw'].shift(24)
data['lag_168'] = data['demand_mw'].shift(168)

In [307]:
data['lag_3'] = data['demand_mw'].shift(3)
data['lag_6'] = data['demand_mw'].shift(6)

In [308]:
data['rolling_mean_24'] = data['demand_mw'].shift(1).rolling(24).mean()
data['rolling_mean_168'] = data['demand_mw'].shift(1).rolling(168).mean()
data['rolling_std_24'] = data['demand_mw'].shift(1).rolling(24).std()

In [309]:
data['target'] = data['demand_mw'].shift(-1)

In [310]:
# keep only numeric
data = data.select_dtypes(include=['number'])

data = data.dropna()

TRAIN / TEST SPLIT

In [311]:
split_year = data.index.year.min()

train = data[data.index.year == split_year]
test = data[data.index.year > split_year]


In [312]:
X_train = train.drop(columns=['target'])
y_train = train['target']

X_test = test.drop(columns=['target'])
y_test = test['target']

In [313]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

mODEL

In [314]:
from sklearn.ensemble import RandomForestRegressor

vmodel = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [315]:
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAPE:", mape)


MAPE: 0.18862480382056776
